# Decoding clinic - **Student edition**

**Objective (what you will achieve):**  
Build a *causal decoding clinic* for Spanish creative generation using a GPT-2 model, and analyze how decoding choices affect output **quality, diversity, stability, and latency**.

**Sub‑objectives:**  
1. Understand practical differences between **top‑k**, **top‑p (nucleus)**, **temperature**, **repetition_penalty**, and **no_repeat_ngram_size** on a causal LM (GPT‑2).
2. Learn to spot and mitigate **degeneration** (loops/repetition).
3. Design a minimal but **reproducible** pipeline: prompts → generations → metrics → plots → qualitative review.
4. Practice safe/robust experiment hygiene: dataset filtering, Drive saving, and HF gated access.

---

## Step‑by‑step plan (read first)

1. **Mount Drive & set folders** *(provided)*  
2. **Login to Hugging Face** *(YOU fill in the token cell)* and (optionally) verify gated repo access.  
3. **Create a prompt CSV** from a HF dataset *(YOU write a few lines)* and save to Drive.  
4. **Prepare prompts**: filter for creative use and normalize *(YOU complete the filter function)*.  
5. **Load GPT‑2 (Spanish)** *(provided)* and define an anti‑degeneration **template** + **stopping criteria** *(YOU add stop tokens)*.  
6. **Run the decoding grid** *(YOU complete the policy list and the grid loop)*; save to both `/content` and Drive.  
7. **Compute metrics & make plots** *(YOU implement the metric functions; scaffolding provided)*.  
8. **Qualitative review**: side‑by‑side outputs *(provided, needs your previous artifacts)*.  
9. **(Optional)** Post‑process to clamp to **120–180 words** *(YOU implement a simple clamping rule)*.


**Cell purpose:** Mount Google Drive, create base folders, and verify write access. *(provided)*

In [ ]:
# DO NOT EDIT
from google.colab import drive
import os, time, glob

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/decoding_clinic"
DATA = f"{BASE}/data/creative"
OUT  = f"{BASE}/outputs/creative_gpt2"

os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

print("BASE:", BASE)
print("DATA:", DATA)
print("OUT :", OUT)

# Write test
test_path = f"{OUT}/__write_test_{int(time.time())}.txt"
with open(test_path, "w") as f:
    f.write("ok")
print("✅ Drive write test:", test_path)
print("Files in OUT:", sorted(glob.glob(f"{OUT}/*"))[:10])


**Cell purpose:** Authenticate to Hugging Face with your own token; optionally verify access to a gated repo. *(YOU fill the token)*

In [ ]:
# TODO: Enter your HF token (private). Do NOT hardcode real tokens in public repos.
!pip -q install huggingface_hub datasets

from huggingface_hub import login, hf_hub_download

HF_TOKEN = "REPLACE_WITH_YOUR_HF_TOKEN"  # TODO
assert HF_TOKEN != "REPLACE_WITH_YOUR_HF_TOKEN", "Please set HF_TOKEN before continuing."
login(token=HF_TOKEN, add_to_git_credential=False)

print("Attempting gated repo access test (optional; may require accepting terms in the Hub):")
gated_repo = "google/gemma-2-2b-it"   # You can change to another gated repo you have access to
try:
    cfg_path = hf_hub_download(repo_id=gated_repo, filename="config.json")
    print("✔ Gated access verified. File:", cfg_path)
except Exception as e:
    print("ℹ️ Could not verify gated access. This is fine if you haven't accepted terms.")
    print("Detail:", str(e))


**Cell purpose:** Load a small public HF dataset and export a `prompt` column to Drive. *(YOU implement minimal logic)*

In [ ]:
# TODO: Load a small dataset split and write DataFrame with a single 'prompt' column to DATA.
from datasets import load_dataset
import pandas as pd, os

DS_ID = "databricks/databricks-dolly-15k"   # You may change dataset if desired
SPLIT = "train[:2%]"                        # Keep small for quick iteration

print("Loading dataset:", DS_ID, SPLIT)
ds = load_dataset(DS_ID, split=SPLIT)

df_src = pd.DataFrame(ds)
# TODO: Map a suitable text column to 'prompt' (e.g., 'instruction').
# Hints: look at df_src.columns; choose a column; cast to string; fillna("");
#        create df_src['prompt']; then export only that column.
# BEGIN STUDENT CODE
chosen_col = "instruction"  # change if needed after inspection
assert chosen_col in df_src.columns, f"Column {chosen_col} not found in dataset."
df_src["prompt"] = df_src[chosen_col].fillna("").astype(str)
# END STUDENT CODE

csv_raw = f"{DATA}/prompts_hf_raw.csv"
df_src[["prompt"]].to_csv(csv_raw, index=False)
print("✅ Saved raw prompts to:", csv_raw)
df_src.head()


**Cell purpose:** Filter out non‑creative prompts (e.g., trivia/QA) and normalize text. *(YOU complete the filter)*

In [ ]:
# TODO: Implement 'is_creative' to drop trivia-like questions that can cause degeneration.
import pandas as pd, re

raw_csv = f"{DATA}/prompts_hf_raw.csv"
df = pd.read_csv(raw_csv)

def is_creative(s: str) -> bool:
    """Return True if 's' looks suitable for creative generation."""
    s = str(s or "").strip()
    # TODO: Tweak this heuristic: drop prompts that start with common QA patterns.
    # BEGIN STUDENT CODE
    return not bool(re.match(r"^(which|when|why|who|where|if|what|how|does|do|is|are|can|could|would|should)\b", s, flags=re.I))
    # END STUDENT CODE

prompts_all = df["prompt"].dropna().astype(str)
prompts_clean = [re.sub(r"[\t\r]+"," ", p).strip() for p in prompts_all if is_creative(p)]

# Provide a small Spanish fallback in case too few remain
synthetic_es = [
    "una llave antigua que abre recuerdos",
    "un tren a medianoche rumbo a lo desconocido",
    "un archivo perdido que nadie debería leer",
    "una cafetería en tormenta y un reencuentro",
    "la ciudad que olvidó cómo soñar",
    "un reloj detenido y una promesa incumplida",
    "un faro que habla con la marea",
    "una carta que llega con diez años de retraso"
]
while len(prompts_clean) < 120:
    prompts_clean.extend(synthetic_es)
prompts = prompts_clean[:120]

print(f"Total creative prompts: {len(prompts)}")
print('Sample:', prompts[:5])

# Save cleaned prompts for traceability
pd.DataFrame({'prompt': prompts}).to_csv(f"{DATA}/prompts_hf_clean.csv", index=False)
print("✅ Saved cleaned prompts:", f"{DATA}/prompts_hf_clean.csv")


**Cell purpose:** Load GPT‑2 (ES), define an anti‑degeneration template, choose stop sequences, and create the generator. *(YOU add stop texts)*

In [ ]:
# Partly provided; YOU add the stop sequences.
!pip -q install transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.generation.stopping_criteria import StoppingCriteriaList, StoppingCriteria
import torch, time, re

MODEL_NAME = "DeepESP/gpt2-spanish"  # You can swap to another Spanish GPT-2 variant
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

class StopOnSeq(StoppingCriteria):
    """Stop generation when any provided stop sequence is matched at the end."""
    def __init__(self, stop_ids):
        super().__init__()
        self.stop_ids = stop_ids
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] < 2:
            return False
        for sid in self.stop_ids:
            if len(sid) <= input_ids.shape[1] and (input_ids[0, -len(sid):] == sid).all():
                return True
        return False

# TODO: Define stop sequences that safely cut off rambling (e.g., double newline, a marker).
# BEGIN STUDENT CODE
stop_texts = ["\n\n", "\nTema:"]  # adjust if needed
# END STUDENT CODE

stop_ids = [tok.encode(t, add_special_tokens=False, return_tensors="pt").to(device)[0] for t in stop_texts]
stops = StoppingCriteriaList([StopOnSeq(stop_ids)])

def apply_template(topic):
    topic = str(topic).replace("\n", " ").strip()
    return (
        "Escribe un texto creativo en español, entre 120 y 180 palabras. "
        "Sé concreto, evita repeticiones, y termina con un punto final.\n"
        f"Tema: {topic}\n"
        "Texto:"
    )

def dedupe_spans(s: str) -> str:
    s = re.sub(r"(\b\w{2,}\b)(?:\s+\1){2,}", r"\1", s, flags=re.I)
    s = re.sub(r'(\"|\“|\”)'+"{2,}", r"\1", s)
    return s

def generate_one(prompt, **gen_kwargs):
    inputs = tok(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    out = model.generate(
        **inputs,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
        stopping_criteria=stops,
        **gen_kwargs
    )
    dt = time.time() - t0
    text = tok.decode(out[0], skip_special_tokens=True)
    text = dedupe_spans(text)
    return text, dt


**Cell purpose:** Define sampling‑focused decoding policies and run the grid; dual‑save CSV to `/content` and Drive. *(YOU complete policies and loop)*

In [ ]:
# TODO: Create a robust sampling grid to reduce repetition; then run it and save results.
import numpy as np, random, pandas as pd, torch, os, glob

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Reasonable default for GPT-2 ES in creative tasks
MAX_NEW_TOKENS = 140

# TODO: Fill/adjust policies: vary top-k, top-p, temperature, repetition_penalty, and no_repeat_ngram_size.
# BEGIN STUDENT CODE
policies = [
    dict(name="topk50_t0.8_rep1.2_blk3",  do_sample=True, top_k=50,  temperature=0.8, repetition_penalty=1.2, no_repeat_ngram_size=3),
    dict(name="topp0.92_t0.8_rep1.2_blk3",do_sample=True, top_p=0.92, temperature=0.8, repetition_penalty=1.2, no_repeat_ngram_size=3),
    dict(name="topp0.9_t0.7_rep1.3_blk4", do_sample=True, top_p=0.90, temperature=0.7, repetition_penalty=1.3, no_repeat_ngram_size=4),
    dict(name="topk100_t0.7_rep1.2_blk3", do_sample=True, top_k=100, temperature=0.7, repetition_penalty=1.2, no_repeat_ngram_size=3),
]
# END STUDENT CODE

def clean_policy_kwargs(pol: dict) -> dict:
    return {k: v for k, v in pol.items() if k != "name"}

subset = prompts[:60]  # You may reduce to [:20] for speed
results = []

# TODO: Implement the generation loop over policies and prompts
# BEGIN STUDENT CODE
for pol in policies:
    print(">> Policy:", pol["name"])
    latencies = []
    for idx, topic in enumerate(subset):
        prompt = apply_template(topic)
        text, dt = generate_one(prompt, max_new_tokens=MAX_NEW_TOKENS, **clean_policy_kwargs(pol))
        results.append({"idx": idx, "policy": pol["name"], "prompt": topic, "output": text, "latency_sec": dt})
        latencies.append(dt)
    print(f"Mean latency: {np.mean(latencies):.3f}s — n={len(latencies)}")
# END STUDENT CODE

df = pd.DataFrame(results)
print("Result columns:", list(df.columns), "| Rows:", len(df))

# Always save to /content
csv_content = "/content/raw_generations_gpt2_auth.csv"
df.to_csv(csv_content, index=False)
print("Saved to /content:", csv_content)

# Try to save to Drive as well
os.makedirs(OUT, exist_ok=True)
csv_out = f"{OUT}/raw_generations_gpt2_auth.csv"
try:
    df.to_csv(csv_out, index=False)
    print("Saved to Drive:", csv_out)
except Exception as e:
    print("⚠️ Could not save to Drive. Use the /content file instead. Reason:", e)


**Cell purpose:** Load results (RAM/Drive/`/content`), compute metrics (distinct/rep/length/latency), save CSVs, and make quick plots. *(YOU implement metric helpers)*

In [ ]:
# TODO: Implement metrics; scaffolding provided.
import pandas as pd, numpy as np, os, glob, re
from collections import Counter
import matplotlib.pyplot as plt

# Prefer df in RAM; otherwise search Drive then /content
use_ram = ('df' in globals()) and isinstance(df, pd.DataFrame) and ('output' in df.columns)
if not use_ram:
    candidates = sorted(glob.glob(f"{OUT}/raw_generations_gpt2*.csv"))
    if not candidates:
        candidates = sorted(glob.glob("/content/raw_generations_gpt2*.csv"))
    assert candidates, "No results found. Re-run the grid."
    latest = candidates[-1]
    print("Loading:", latest)
    df = pd.read_csv(latest)
    assert "output" in df.columns, "CSV lacks 'output'. Re-run the grid."

# TODO: Implement tokenization for metrics
# BEGIN STUDENT CODE
def words(text):
    return re.findall(r"\w+|[\.,;:!?¡¿()\"'—-]", str(text).lower())

def ngrams(seq, n):
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)] if len(seq) >= n else []

def diversity_metrics(text):
    ws = words(text)
    ng2 = ngrams(ws, 2); ng3 = ngrams(ws, 3)
    def distinct(ng): return len(set(ng))/max(1,len(ng))
    def rep(ng):
        c = Counter(ng); return (sum(v-1 for v in c.values() if v>1))/max(1,len(ng))
    return {"len_words": len(ws), "distinct2": distinct(ng2), "distinct3": distinct(ng3),
            "rep2": rep(ng2), "rep3": rep(ng3)}
# END STUDENT CODE

# Compute per-example metrics
mrows = [{**row.to_dict(), **diversity_metrics(row["output"])} for _, row in df.iterrows()]
mdf = pd.DataFrame(mrows)

# Aggregate by policy
agg = mdf.groupby("policy")["len_words","distinct2","distinct3","rep2","rep3","latency_sec"].agg(
    ["mean","std","median"]
).reset_index()
agg.columns = ["policy"] + [f"{a}_{b}" for a,b in agg.columns.tolist()[1:]]

# Save metrics if Drive exists
if os.path.exists(OUT):
    csv_sum = f"{OUT}/metrics_summary_gpt2_auth.csv"
    csv_per = f"{OUT}/metrics_per_example_gpt2_auth.csv"
    agg.to_csv(csv_sum, index=False); mdf.to_csv(csv_per, index=False)
    print("Saved:", csv_sum, "|", csv_per)
else:
    print("Drive OUT not found; showing metrics without saving.")

# Quick plots
def bar_from_summary(metric):
    order = agg.sort_values(f"{metric}_mean")["policy"].tolist()
    vals = agg.set_index("policy")[f"{metric}_mean"].loc[order].values
    errs = agg.set_index("policy")[f"{metric}_std"].loc[order].values
    plt.figure(figsize=(10,4))
    plt.bar(order, vals, yerr=errs)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"{metric} (mean ± std) across policies")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()

bar_from_summary("distinct2")
bar_from_summary("distinct3")
bar_from_summary("latency_sec")

plt.figure(figsize=(10,4))
data = [mdf.loc[mdf.policy==p,"len_words"].values for p in agg["policy"]]
plt.boxplot(data, labels=agg["policy"], showmeans=True)
plt.xticks(rotation=30, ha="right")
plt.title("Length distribution (words) by policy")
plt.tight_layout()
plt.show()

agg.head()


**Cell purpose:** Show a side‑by‑side table of outputs by policy for selected prompts. *(provided)*

In [ ]:
# DO NOT EDIT
import pandas as pd
SAMPLES_TO_SHOW = [0, 1, 2]
policies_order = agg["policy"].tolist()

for idx in SAMPLES_TO_SHOW:
    display(pd.DataFrame({
        "policy": policies_order,
        "output": [df[(df.idx==idx)&(df.policy==p)].iloc[0]["output"] for p in policies_order]
    }))


**Cell purpose (optional):** Softly clamp outputs to **120–180 words** and export an additional CSV. *(YOU implement a simple rule)*

In [ ]:
# TODO: Implement a simple clamping rule; export to Drive and /content.
import re, pandas as pd, os

# BEGIN STUDENT CODE
def word_list(txt):
    return re.findall(r"\w+|[\.,;:!?¡¿()\"'—-]", str(txt))

def clamp_to_range(text, lo=120, hi=180):
    toks = word_list(text)
    if lo <= len(toks) <= hi:
        return text
    if len(toks) > hi:
        cut = " ".join(toks[:hi])
        m = re.search(r"^(.*?[\.!?])(.*)$", cut)
        return m.group(1) if m else cut
    return text  # do not pad if short
# END STUDENT CODE

df_len = df.copy()
df_len["output_len_clamped"] = df_len["output"].apply(lambda t: clamp_to_range(t, 120, 180))

csv_len_content = "/content/raw_generations_gpt2_auth_lenclamped.csv"
df_len.to_csv(csv_len_content, index=False)
print("Saved to /content:", csv_len_content)

if os.path.exists(OUT):
    csv_len_drive = f"{OUT}/raw_generations_gpt2_auth_lenclamped.csv"
    df_len.to_csv(csv_len_drive, index=False)
    print("Saved to Drive:", csv_len_drive)
